# ReFinED architecture with random weights

RoBERTa context encoding, BIO detection, mean span pooling, typing, a two-layer description Transformer and candidate scoring follow [ReFinED 7f57bd2](https://github.com/amazon-science/ReFinED/tree/7f57bd2348bc023793dd1cb93818757a7f33cc4a/src/refined/model_components).
Only tokenizer/configuration files are loaded; all model weights are random. The small KB and gold spans are demo fixtures. Explicit candidate validity preserves real candidates with zero prior.


## Complete flow: function → input → output

| Function | Input | Output |
|---|---|---|
| `tokenizer.encode_text` | Text | IDs, attention mask, character offsets |
| `context_encoder` | IDs + mask | Contextual token vectors |
| `mention_detection` | Token vectors | BIO logits and optional loss |
| `decode_mentions` | BIO labels, offsets, vectors | Texts, spans, mean mention vectors |
| `entity_typing` | Mention vectors | Independent type probabilities |
| `retriever` | Mention texts | Candidate records, known types, priors |
| `tokenizer.encode_descriptions` | Labels + descriptions | Paired token IDs grouped by mention/candidate |
| `description_encoder` | Candidate token IDs | 300-dimensional vectors |
| `description_matching` | Mention + description vectors | Candidate/NIL probabilities |
| `entity_disambiguation` | Types, priors, description matches, validity mask | Candidate/NIL scores |
| `argmax` | Scores | Selected entity or `None` |


In [31]:
from copy import deepcopy

import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from transformers import AutoTokenizer, RobertaConfig, RobertaModel
from IPython.display import display

torch.manual_seed(7)
torch.set_num_threads(2)


def init_weights(module):
    """ReFinED initializes linear/embedding weights with std=0.02."""
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
    if isinstance(module, nn.Linear) and module.bias is not None:
        nn.init.zeros_(module.bias)


class LinkingTokenizer:
    def __init__(self, backbone_name):
        self.tokenizer = AutoTokenizer.from_pretrained(backbone_name, use_fast=True, add_prefix_space=False)

    def encode_text(self, text):
        return self.tokenizer(text, return_offsets_mapping=True, return_tensors="pt")

    def encode_descriptions(self, labels, descriptions, num_mentions, num_candidates, max_length=32):
        if num_mentions < 1 or num_candidates < 1 or len(labels) != num_mentions * num_candidates or len(descriptions) != len(labels):
            raise ValueError("Provide one label/description pair per candidate in mention order.")
        tokens = self.tokenizer(
            labels, text_pair=descriptions, padding="max_length", truncation=True,
            max_length=max_length, return_tensors="pt",
        )
        return tokens["input_ids"].reshape(num_mentions, num_candidates, max_length)


backbone_name = "FacebookAI/roberta-base"
tokenizer = LinkingTokenizer(backbone_name)
config = RobertaConfig.from_pretrained(backbone_name)


/mnt/storage/projects/learning-llm-components/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## 1. Context encoding, mention detection and pooling

The BIO head sees contextual vectors. Gold spans bypass its random predictions for the demo; both paths use the same decoder. An orphan `I` starts a mention, as in upstream decoding.


In [32]:
class MentionDetection(nn.Module):
    def __init__(self, hidden_size, num_labels=3, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.linear = nn.Linear(hidden_size, num_labels)
        self.apply(init_weights)

    def forward(self, contextualised_embeddings, ner_labels=None):
        logits = self.linear(self.dropout(contextualised_embeddings))
        loss = None if ner_labels is None else F.cross_entropy(
            logits.reshape(-1, logits.size(-1)), ner_labels.reshape(-1), ignore_index=-100,
        )
        return loss, logits


def decode_mentions(text, labels, offsets, token_vectors):
    """Decode one sequence and mean-pool each mention, preserving source order."""
    token_spans = []
    start = None
    for index, label in enumerate(labels.tolist()):
        if offsets[index, 1] <= offsets[index, 0]:
            label = 0
        if start is not None and label in (0, 1):
            token_spans.append((start, index))
            start = None
        if start is None and label != 0:
            start = index
    if start is not None:
        token_spans.append((start, len(labels)))
    spans = [(offsets[a, 0].item(), offsets[b - 1, 1].item()) for a, b in token_spans]
    vectors = torch.stack([token_vectors[a:b].mean(0) for a, b in token_spans]) if token_spans else token_vectors.new_empty((0, token_vectors.size(-1)))
    return vectors, [text[a:b] for a, b in spans], spans


# Constructors initialize random weights; no pretrained model weights are loaded.
context_encoder = RobertaModel(config, add_pooling_layer=False).eval()
mention_detection = MentionDetection(config.hidden_size).eval()
mention_dropout = nn.Dropout(0.1).eval()
text = "Hawaii and Paris are places."
inputs = tokenizer.encode_text(text)
offsets = inputs["offset_mapping"][0]

# Align the fixture spans with tokenizer offsets, including multi-token names.
gold_spans = [(0, 6), (11, 16)]
smoke_bio_labels = torch.zeros_like(inputs["input_ids"])
for start, end in gold_spans:
    indices = ((offsets[:, 0] < end) & (offsets[:, 1] > start)).nonzero().flatten()
    assert indices.numel() and offsets[indices[0], 0] == start and offsets[indices[-1], 1] == end
    smoke_bio_labels[0, indices] = 2
    smoke_bio_labels[0, indices[0]] = 1

with torch.no_grad():
    token_vectors = context_encoder(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"]).last_hidden_state
    _, mention_logits = mention_detection(token_vectors)
    bio_labels = smoke_bio_labels  # Use mention_logits.argmax(-1) for random model predictions.
    mention_vectors, mention_texts, mention_spans = decode_mentions(text, bio_labels[0], offsets, token_vectors[0])
    mention_vectors = mention_dropout(mention_vectors)

mention_table = pd.DataFrame(mention_vectors.numpy()[:, :8]).add_prefix("feature_")
mention_table.insert(0, "span", mention_spans)
mention_table.insert(0, "mention", mention_texts)
display(mention_table)


,mention,span,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7
0,Hawaii,"(0, 6)",1.810813,0.699508,-0.033150,0.496361,-1.336091,0.917056,0.410868,-0.554222
1,Paris,"(11, 16)",1.815521,0.708024,1.107395,0.349371,-0.433868,1.036608,0.388564,0.495471


## 2. Entity typing: one probability per type

Predict categories from the mention vectors. The shared `type_names` order also defines the KB type-vector columns used by the next step.

In [33]:
# Upstream reserves type index 0 for padding.
type_names = ["padding", "place", "state", "island"]


class EntityTyping(nn.Module):
    def __init__(self, num_classes, encoder_hidden_size):
        super().__init__()
        self.linear = nn.Linear(encoder_hidden_size, num_classes)
        self.apply(init_weights)

    def forward(self, mention_embeddings, span_classes=None):
        logits = self.linear(mention_embeddings)
        loss = None if span_classes is None else F.binary_cross_entropy_with_logits(logits, span_classes)
        return loss, logits.sigmoid()


entity_typing = EntityTyping(len(type_names), config.hidden_size).eval()
with torch.no_grad():
    _, type_probabilities = entity_typing(mention_vectors)
type_table = pd.DataFrame(type_probabilities.numpy(), columns=type_names)
type_table.insert(0, "mention", mention_texts)
display(type_table)


,mention,padding,place,state,island
0,Hawaii,0.637382,0.256939,0.591532,0.493346
1,Paris,0.569704,0.262934,0.607480,0.499026


## 3. Candidate retrieval

Match names exactly and select two candidates per mention by prior. Keep mention order and original prior values. Missing or insufficient candidates raise an error; every returned slot is valid, including candidates with zero prior.


In [34]:
from pathlib import Path

# Mock candidates with invented priors; supports running from the module or repo root.
module_dir = Path.cwd() if (Path.cwd() / "models.ipynb").is_file() else Path.cwd() / "other_entity_linking"
kb = pd.read_csv(module_dir / "assets/kb.csv")
display(kb)


,mention,entity_id,label,description,prior,padding,place,state,island
0,Hawaii,Q782,Hawaii,state of the United States of America,0.85,0.0,1.0,1.0,0.0
1,Hawaii,Q68740,Hawaii,largest of the Hawaiian islands,0.15,0.0,1.0,0.0,1.0
2,Paris,Q90,Paris,capital of France,0.90,0.0,1.0,0.0,0.0
3,Paris,Q830149,Paris,city in Texas,0.10,0.0,1.0,0.0,0.0


In [35]:
class CandidateRetriever:
    """Retrieve the highest-prior candidates for each mention, preserving mention order."""
    def __init__(self, kb: pd.DataFrame, type_names: list[str], num_candidates: int = 2):
        if not isinstance(num_candidates, int) or isinstance(num_candidates, bool) or num_candidates < 1:
            raise ValueError("num_candidates must be a positive integer.")
        self.kb = kb
        self.type_names = type_names
        self.num_candidates = num_candidates

    def __call__(self, mention_texts: list[str]) -> tuple[
        list[list[str]], pd.DataFrame, torch.Tensor, torch.Tensor
    ]:
        """Return IDs [M,C], records [M*C rows], known types [M,C,T], priors [M,C]."""
        num_mentions = len(mention_texts)
        if num_mentions == 0:
            raise ValueError("Provide at least one mention for this retrieval demo.")

        mention_rows = pd.DataFrame({"mention_index": range(num_mentions), "mention": mention_texts})
        records = mention_rows.merge(self.kb, on="mention", how="left", sort=False)
        if records["entity_id"].isna().any():
            raise ValueError("A mention has no candidates in the mock KB.")
        if records.groupby("mention_index").size().min() < self.num_candidates:
            raise ValueError("A mention has fewer KB candidates than requested; reduce num_candidates.")

        # Rank candidates within each mention.
        records = records.sort_values(["mention_index", "prior"], ascending=[True, False], kind="stable")
        records = records.groupby("mention_index", sort=False).head(self.num_candidates).reset_index(drop=True)

        candidate_ids = records["entity_id"].to_numpy().reshape(num_mentions, self.num_candidates).tolist()
        known_types = torch.tensor(records[self.type_names].to_numpy(), dtype=torch.float32)
        known_types = known_types.reshape(num_mentions, self.num_candidates, len(self.type_names))
        priors = torch.tensor(records["prior"].to_numpy(), dtype=torch.float32)
        priors = priors.reshape(num_mentions, self.num_candidates)
        return candidate_ids, records, known_types, priors


retriever = CandidateRetriever(kb=kb, type_names=type_names, num_candidates=2)
candidate_ids, candidate_records, known_types, priors = retriever(mention_texts)
num_mentions, num_candidates = priors.shape
# The retriever returns only real candidates, with no padded slots.
candidate_mask = torch.ones_like(priors, dtype=torch.bool)

display(candidate_records)


,mention_index,mention,entity_id,label,description,prior,padding,place,state,island
0,0,Hawaii,Q782,Hawaii,state of the United States of America,0.85,0.0,1.0,1.0,0.0
1,0,Hawaii,Q68740,Hawaii,largest of the Hawaiian islands,0.15,0.0,1.0,0.0,1.0
2,1,Paris,Q90,Paris,capital of France,0.90,0.0,1.0,0.0,0.0
3,1,Paris,Q830149,Paris,city in Texas,0.10,0.0,1.0,0.0,0.0


## 4. Description encoder

Tokenize labels and descriptions as pairs, pad to 32 tokens, then project the first-token representation from a two-layer RoBERTa encoder to 300 dimensions. Its embedding layer is frozen, matching upstream.


In [36]:
description_ids = tokenizer.encode_descriptions(
    candidate_records["label"].tolist(), candidate_records["description"].tolist(),
    num_mentions, num_candidates,
)


In [37]:
class DescriptionEncoder(nn.Module):
    def __init__(self, config, n_layer=2, output_dim=300, dropout=0.1):
        super().__init__()
        description_config = deepcopy(config)
        description_config.num_hidden_layers = n_layer
        description_config.chunk_size_feed_forward = 4
        self.transformer = RobertaModel(description_config, add_pooling_layer=False)
        self.transformer.embeddings.requires_grad_(False)
        self.dropout = nn.Dropout(dropout)
        self.projection = nn.Linear(config.hidden_size, output_dim)
        self.projection.apply(init_weights)
        self.pad_token_id = config.pad_token_id

    def forward(self, input_ids):
        num_mentions, num_candidates, length = input_ids.shape
        flat_ids = input_ids.reshape(-1, length)
        valid = flat_ids[:, 0].ne(self.pad_token_id)
        vectors = self.projection.weight.new_zeros((flat_ids.size(0), self.projection.in_features))
        if valid.any():
            ids = flat_ids[valid]
            vectors[valid] = self.transformer(
                input_ids=ids, attention_mask=ids.ne(self.pad_token_id),
            ).last_hidden_state[:, 0]
        return self.projection(self.dropout(vectors)).reshape(num_mentions, num_candidates, -1)


description_encoder = DescriptionEncoder(config).eval()
with torch.no_grad():
    description_vectors = description_encoder(description_ids)
print("Description vectors:", description_vectors.shape)
description_table = candidate_records[["mention", "entity_id", "label", "description"]].copy()
display(description_table)


Description vectors: torch.Size([2, 2, 300])


,mention,entity_id,label,description
0,Hawaii,Q782,Hawaii,state of the United States of America
1,Hawaii,Q68740,Hawaii,largest of the Hawaiian islands
2,Paris,Q90,Paris,capital of France
3,Paris,Q830149,Paris,city in Texas


## 5. Description matching: one probability per candidate, plus NIL

In [38]:
class EDLayer(nn.Module):
    def __init__(self, description_encoder, mention_dim=768, output_dim=300):
        super().__init__()
        self.description_encoder = description_encoder
        self.mention_projection = nn.Linear(mention_dim, output_dim)
        self.mention_projection.apply(init_weights)

    def forward(self, mention_embeddings, candidate_desc, candidate_entity_targets=None, candidate_desc_emb=None):
        projected = self.mention_projection(mention_embeddings)
        vectors = self.description_encoder(candidate_desc) if candidate_desc_emb is None else candidate_desc_emb
        scores = (vectors @ projected.unsqueeze(-1)).squeeze(-1)
        # Upstream masks missing descriptions separately from candidate validity.
        valid = candidate_desc[:, :, 0].ne(self.description_encoder.pad_token_id) if candidate_desc_emb is None else candidate_desc_emb[:, :, 0].ne(0)
        scores = scores.masked_fill(~valid, -100)
        scores = torch.cat((scores, scores.new_zeros((scores.size(0), 1))), dim=-1)
        loss = None
        if candidate_entity_targets is not None:
            targets = candidate_entity_targets.argmax(-1)
            missing = scores[torch.arange(scores.size(0), device=scores.device), targets].eq(-100)
            targets = targets.masked_fill(missing, scores.size(-1) - 1)
            loss = F.cross_entropy(scores, targets)
        return loss, scores.softmax(-1)


description_matching = EDLayer(description_encoder, mention_dim=config.hidden_size).eval()
with torch.no_grad():
    _, description_probabilities = description_matching(mention_vectors, description_ids)

score_entity_ids = [entity for ids in candidate_ids for entity in ids + ["NIL"]]
score_mentions = [mention for mention in mention_texts for _ in range(num_candidates + 1)]
match_table = pd.DataFrame({"mention": score_mentions, "entity_id": score_entity_ids,
                            "description_match": description_probabilities.flatten().tolist()})
display(match_table)


,mention,entity_id,description_match
0,Hawaii,Q782,0.000644
1,Hawaii,Q68740,0.001081
2,Hawaii,NIL,0.998275
3,Paris,Q90,0.000363
4,Paris,Q830149,0.000336
5,Paris,NIL,0.999301


## 6. Entity disambiguation

Score each candidate using per-type agreement, prior, type distance and description probability. Append a fixed NIL score of zero.


In [39]:
class EntityDisambiguation(nn.Module):
    def __init__(self, num_classes, dropout=0.05):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(num_classes + 3, 1)
        self.apply(init_weights)

    def candidate_features(self, class_activations, candidate_pem_values, candidate_classes, candidate_description_scores):
        predicted = class_activations.unsqueeze(1)
        agreement = candidate_classes * predicted
        distance = (candidate_classes - predicted).square().sum(-1, keepdim=True).sqrt()
        return torch.cat((agreement, candidate_pem_values.unsqueeze(-1), distance,
                          candidate_description_scores[:, :-1].unsqueeze(-1)), dim=-1)

    def forward(self, class_activations, candidate_pem_values, candidate_classes,
                candidate_description_scores, candidate_mask, candidate_entity_targets=None):
        features = self.candidate_features(class_activations, candidate_pem_values, candidate_classes, candidate_description_scores)
        scores = self.classifier(self.dropout(features)).squeeze(-1)
        # Intentional fix: zero prior does not mean the candidate is padding.
        scores = scores.masked_fill(~candidate_mask, -1e8)
        scores = torch.cat((scores, scores.new_zeros((scores.size(0), 1))), dim=1)
        loss = None if candidate_entity_targets is None else F.cross_entropy(scores, candidate_entity_targets.argmax(-1))
        return loss, scores


entity_disambiguation = EntityDisambiguation(len(type_names)).eval()
features = entity_disambiguation.candidate_features(type_probabilities, priors, known_types, description_probabilities)
columns = [name + "_agreement" for name in type_names] + ["prior", "type_distance", "description_match"]
evidence_table = pd.DataFrame(features.reshape(-1, features.size(-1)).numpy(), columns=columns)
evidence_table.insert(0, "entity_id", candidate_records["entity_id"].tolist())
evidence_table.insert(0, "mention", candidate_records["mention"].tolist())
display(evidence_table)

with torch.no_grad():
    _, entity_scores = entity_disambiguation(
        class_activations=type_probabilities.detach(), candidate_pem_values=priors,
        candidate_classes=known_types, candidate_description_scores=description_probabilities.detach(),
        candidate_mask=candidate_mask,
    )
assert entity_scores.shape == (num_mentions, num_candidates + 1)


,mention,entity_id,padding_agreement,place_agreement,state_agreement,island_agreement,prior,type_distance,description_match
0,Hawaii,Q782,0.0,0.256939,0.591532,0.000000,0.85,1.169885,0.000644
1,Hawaii,Q68740,0.0,0.256939,0.000000,0.493346,0.15,1.251001,0.001081
2,Paris,Q90,0.0,0.262934,0.000000,0.000000,0.90,1.218970,0.000363
3,Paris,Q830149,0.0,0.262934,0.000000,0.000000,0.10,1.218970,0.000336


## 7. Select the highest score; untrained scores have no factual meaning

In [40]:
winning_columns = entity_scores.argmax(dim=1).tolist()
selected_ids = [None if column == num_candidates else ids[column]
                for ids, column in zip(candidate_ids, winning_columns)]
results = pd.DataFrame({
    "mention": mention_texts,
    "start": [start for start, end in mention_spans],
    "end": [end for start, end in mention_spans],
    "entity_id": selected_ids,
    "untrained_demo": True,
})
score_table = pd.DataFrame({"mention": score_mentions, "entity_id": score_entity_ids,
                            "score": entity_scores.flatten().tolist()})
score_table["selected"] = torch.arange(num_candidates + 1).unsqueeze(0).eq(entity_scores.argmax(dim=1, keepdim=True)).flatten().tolist()
display(score_table)
display(results)


,mention,entity_id,score,selected
0,Hawaii,Q782,-0.026384,False
1,Hawaii,Q68740,-0.016887,False
2,Hawaii,NIL,0.000000,True
3,Paris,Q90,-0.026127,False
4,Paris,Q830149,-0.026665,False
5,Paris,NIL,0.000000,True


,mention,start,end,entity_id,untrained_demo
0,Hawaii,0,6,None,True
1,Paris,11,16,None,True


## Weaknesses

| Weakness | What happens | Fix |
|---|---|---|
| Random weights | Scores have no linking accuracy claim | Train the model before using predictions |
| Gold fixture spans | The demo bypasses predicted BIO labels | Evaluate detection separately with labeled examples |
| Mock KB | Two candidates and three real types replace upstream lookups | Supply real candidate priors, descriptions and type vectors |
| Single-sequence example | No document chunking or upstream word-level BIO reconciliation | Restore upstream preprocessing for long documents |

Description tokenization, encoder dimensions, head initialization, dropout and inference equations follow the pinned source. Optional training losses are exposed, but there is no training loop. Candidate validity intentionally uses an explicit mask instead of upstream's zero-prior padding convention. Random encoders are independently initialized rather than loaded from pretrained weights.
